In [1]:
# !pip install diffusers
from diffusers import DDPMPipeline, DDIMPipeline, PNDMPipeline

model_id = "google/ddpm-cifar10-32"

# load model and scheduler
ddpm = DDPMPipeline.from_pretrained(model_id)  # you can replace DDPMPipeline with DDIMPipeline or PNDMPipeline for faster inference

# run pipeline in inference (sample random noise and denoise)
image = ddpm().images[0]

# save image
image.save("ddpm_generated_image.png")


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
import streamlit as st
import torch
from diffusers import StableDiffusionPipeline
from io import BytesIO

# ---------------------------------------------------------
# Streamlit page configuration
# ---------------------------------------------------------

st.set_page_config(
    page_title="AI Image Generator",
    page_icon="🎨",
    layout="centered"
)

st.title("🎨 AI Text-to-Image Generator")
st.write("Enter a text prompt and generate an image using Stable Diffusion.")

# ---------------------------------------------------------
# Device configuration
# ---------------------------------------------------------

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

st.info(f"Using device: {device}")

# ---------------------------------------------------------
# Load Stable Diffusion model
# ---------------------------------------------------------

@st.cache_resource
def load_model():

    model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype
    )

    pipe = pipe.to(device)

    # Optional memory optimization
    if device == "cuda":
        pipe.enable_attention_slicing()

    return pipe


# Load model
pipe = load_model()

# ---------------------------------------------------------
# User input
# ---------------------------------------------------------

prompt = st.text_area(
    "Enter your image prompt:",
    placeholder="Example: A futuristic city at sunset, cinematic lighting, highly detailed",
    height=100
)

negative_prompt = st.text_input(
    "Negative prompt (optional):",
    placeholder="blurry, low quality, distorted"
)

# ---------------------------------------------------------
# Generation settings
# ---------------------------------------------------------

col1, col2 = st.columns(2)

with col1:
    steps = st.slider(
        "Inference steps",
        min_value=10,
        max_value=50,
        value=25
    )

with col2:
    guidance = st.slider(
        "Guidance scale",
        min_value=1.0,
        max_value=15.0,
        value=7.5,
        step=0.5
    )

# ---------------------------------------------------------
# Generate image
# ---------------------------------------------------------

if st.button("🚀 Generate Image", type="primary"):

    if not prompt.strip():
        st.warning("Please enter a prompt first.")

    else:

        with st.spinner("Generating image..."):

            try:

                result = pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=steps,
                    guidance_scale=guidance
                )

                image = result.images[0]

                # Store image in session state
                st.session_state["generated_image"] = image

            except Exception as e:

                st.error(f"Error while generating image: {e}")


# ---------------------------------------------------------
# Display generated image
# ---------------------------------------------------------

if "generated_image" in st.session_state:

    image = st.session_state["generated_image"]

    st.subheader("Generated Image")

    st.image(
        image,
        caption=prompt,
        use_container_width=True
    )

    # -----------------------------------------------------
    # Convert image to bytes for download
    # -----------------------------------------------------

    image_bytes = BytesIO()

    image.save(
        image_bytes,
        format="PNG"
    )

    image_bytes.seek(0)

    # -----------------------------------------------------
    # Download button
    # -----------------------------------------------------

    st.download_button(
        label="⬇️ Download Image",
        data=image_bytes,
        file_name="generated_image.png",
        mime="image/png"
    )


2026-09-30 07:39:45.229 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-30 07:39:45.235 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-30 07:39:45.652 
  command:

    streamlit run C:\python\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-30 07:39:45.652 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-30 07:39:45.656 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-30 07:39:45.656 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-30 07:39:45.659 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mod

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

C:\python\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]